In [1]:
import pandas as pd
import numpy as np
import torch
import os
import re
import nltk
from degender_pronoun import degenderizer

download_dir = os.path.join('..', 'data', 'nltk_data')
os.makedirs(download_dir, exist_ok=True)
nltk.download('punkt_tab', download_dir=download_dir)
nltk.data.path.append(download_dir)

[nltk_data] Downloading package punkt_tab to ../data/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
df1 = pd.read_csv('../data/sentence_sets_trimmed.csv', encoding='mac-roman')
df1 = df1[df1['full_text_tokens'] > 10]

In [3]:
df2 = pd.read_csv('../data/letters_2021.csv')

In [4]:
degender_pronouns = {
    ' mr ': ' mx ',
    ' mrs ': ' mx ',
    ' ms ': ' mx ',
    ' miss ': ' mx ',
    ' mister ': ' mx ',
    ' ms. ': 'mx ',
    ' mr. ': 'mx ',
    ' mrs. ': ' mx '
}

degender_nouns = {
    ' man ': ' person ',
    ' men ': ' persons ',
    ' woman ': ' person ',
    ' women ': ' persons ',
    " man's ": " person's",
    " men's ": " person's",
    " woman's ": " person's",
    " women's ": " person's",
    " gentleman ": " person ",
    " lady ": " person ",
    " gentleman's ": " person's ",
    " lady's ": " person's ",
}

In [5]:
def preprocess(df, data_column, preprocess_type):
    if preprocess_type == 'none':
        return df

    D = degenderizer()
    df[data_column] = df[data_column].apply(lambda x: D.degender(x) if len(x) > 5 else x)

    for k, v in degender_pronouns.items():
        df[data_column] = df[data_column].str.lower().replace(k,v)

    if preprocess_type == 'all':
        for k, v in degender_nouns.items():
            df[data_column] = df[data_column].str.lower().replace(k,v)
    return df

# Preprocess Data

### First Dataset

In [6]:
df1 = preprocess(df1, 's1_s2', 'all')

In [7]:
df1 = preprocess(df1, 'full_text', 'all')

In [8]:
# df1.to_csv('../data/sentence_sets_trimmed_processed.csv', index=False)

### Second Dataset

In [9]:
df2['sentences'] = df2['LETTERTEXT'].apply(nltk.sent_tokenize)

In [10]:
identifier_pattern = r'(?:^|\b|[^\w\s]+)identifier(?:\b|[^\w\s]+|$)'
pronoun_pattern = r"(?:^|\b|[^\w\s]+)(he|she|him|himself|herself|her|his|hers|he’s|she’s|he's|she's)(?:\b|[^\w\s]+|$)"

In [11]:
def classify_sentence(sentence):
    if re.search(identifier_pattern, sentence, re.IGNORECASE):
        return 's1'
    elif re.search(pronoun_pattern, sentence, re.IGNORECASE):
        return 's2'
    else:
        return 's3'

In [12]:
df2['s1'] = None
df2['s2'] = None
df2['s3'] = None
df2['s1_s2'] = None

for idx, row in df2.iterrows():
    s1_sentences = []
    s2_sentences = []
    s3_sentences = []
    s1_s2_sentences = []
    
    for sentence in row['sentences']:
        category = classify_sentence(sentence)
        if category == 's1':
            s1_sentences.append(sentence)
            s1_s2_sentences.append(sentence)
        elif category == 's2':
            s2_sentences.append(sentence)
            s1_s2_sentences.append(sentence)
        else:
            s3_sentences.append(sentence)
    
    df2.at[idx, 's1'] = ' '.join(s1_sentences)
    df2.at[idx, 's2'] = ' '.join(s2_sentences)
    df2.at[idx, 's3'] = ' '.join(s3_sentences)
    df2.at[idx, 's1_s2'] = ' '.join(s1_s2_sentences)

In [13]:
df2 = preprocess(df2, 's1_s2', 'all')

In [ ]:
df2 = preprocess(df2, 'LETTERTEXT', 'all')

In [14]:
# df2.to_csv('../data/letters_2021_processed.csv', index=False)